In [ ]:
from pdf2image import convert_from_path
import google.generativeai as genai
import pytesseract
import time
import json

In [ ]:
path_to_tesseract_exe = ''
pdf_path = ''
poppler_path = ''
pytesseract.pytesseract.tesseract_cmd = path_to_tesseract_exe

pages = convert_from_path(
    pdf_path,
    poppler_path=poppler_path
)
full_text = ''
for i, page in enumerate(pages):
    text = pytesseract.image_to_string(page, lang="eng")
    full_text += text

In [70]:
def chunk_text(text):
    text_len = 500
    ls = []
    iter = len(text)//text_len
    for i in range(0, len(text), text_len):
        chunk = text[i:i+text_len]
        ls.append(chunk)
    return ls
chunked = chunk_text(full_text)

In [ ]:
API_KEY = ""

genai.configure(api_key=API_KEY)

model = genai.GenerativeModel('gemini-2.5-flash')

In [76]:
instruction_dataset = []

for chunk in chunked:
    prompt = f"Summarize the following text in uzbek:\n\n{chunk}"
    response = model.generate_content(prompt)
    time.sleep(20)
    item = {
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": f"Summarize the following text in uzbek:\n\n{chunk}"},
            {"role": "assistant", "content": response.text}
        ]
    }
    
    instruction_dataset.append(item)

In [94]:
with open("summurize_dataset.jsonl", "w", encoding="utf-8") as f:
    for item in instruction_dataset:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

In [99]:
instruction_dataset_qa = []

for chunk in chunked:
    prompt = f"""
        You are generating a high-quality question-answer dataset.

        For each text below:
        - Create 1 meaningful question
        - Provide a precise answer based only on the text
        - Return ONLY valid JSON
        - Do NOT use markdown.
        - Do NOT wrap the output in ```json.
        - Output format:

        [
        {{
            "question": "...",
            "answer": "..."
        }}
        ]

        Texts:
        {chunk}
        """
    response = model.generate_content(prompt)
    time.sleep(20)
    instruction_dataset_qa.append(response.text)

In [102]:
with open("instruction_dataset_qa.jsonl", "w", encoding="utf-8") as f:
    for item in instruction_dataset_qa:
        data = json.loads(item)
        f.write(json.dumps(data, ensure_ascii=False) + "\n")

In [103]:
len(full_text)

4658

In [104]:
len(chunked)

10

In [105]:
full_text

'ADABIYOT MEHROBIDAGI ABADIYAT\nZamonning talabi katta, shiddati esa tez. Bugunning yangi\nkashfiyotidan hayratga to‘la taassurotingiz arimay turib boshga\nbir yangilik sizni o‘z sirliligiga tortishi ham bor gap. Albatta,\nyangi-yangi ixtirolar, hayotimizga kirib kelayotgan turli texnika\nositalari mehnatimizni yengillashtirishi, har jabhada bizning\nimkoniyatlarimizni oshirishi ayni haqigat. Ammo jamiyat hayo-\ntida qanchadan qancha kashfiyotlar qilinmasin inson ma’naviy\ndunyosining rivojida kitobning o‘rni beqiyos. Kishi ruhiyatining\nma’nan boyishida, xalqparvarlik, diyonat, olijanoblik, insoniy\nfazilatlarning kamol topishida hech bir zamon ixtirosi adabiyot-\nchalik kuchga ega emas. Zero, yurtboshimiz Shavkat\nMirziyoyevning 2017-yil 13-sentabrdagi «Kitob mahsulot-\nlarini nashr etish va targatish tizimini rivojlantirish, kitob\nmutolaasi va kitobxonlik madaniyatini oshirish hamda tar-\ng‘ib qilish bo‘yicha kompleks choratadbirlar dasturi to‘g‘ri-\nsidagi» Qarori ham hozirgi kuni

In [106]:
chunked

['ADABIYOT MEHROBIDAGI ABADIYAT\nZamonning talabi katta, shiddati esa tez. Bugunning yangi\nkashfiyotidan hayratga to‘la taassurotingiz arimay turib boshga\nbir yangilik sizni o‘z sirliligiga tortishi ham bor gap. Albatta,\nyangi-yangi ixtirolar, hayotimizga kirib kelayotgan turli texnika\nositalari mehnatimizni yengillashtirishi, har jabhada bizning\nimkoniyatlarimizni oshirishi ayni haqigat. Ammo jamiyat hayo-\ntida qanchadan qancha kashfiyotlar qilinmasin inson ma’naviy\ndunyosining rivojida kitobning ',
 'o‘rni beqiyos. Kishi ruhiyatining\nma’nan boyishida, xalqparvarlik, diyonat, olijanoblik, insoniy\nfazilatlarning kamol topishida hech bir zamon ixtirosi adabiyot-\nchalik kuchga ega emas. Zero, yurtboshimiz Shavkat\nMirziyoyevning 2017-yil 13-sentabrdagi «Kitob mahsulot-\nlarini nashr etish va targatish tizimini rivojlantirish, kitob\nmutolaasi va kitobxonlik madaniyatini oshirish hamda tar-\ng‘ib qilish bo‘yicha kompleks choratadbirlar dasturi to‘g‘ri-\nsidagi» Qarori ham hozirg

In [107]:
len(instruction_dataset_qa)

10

In [108]:
instruction_dataset_qa

['[\n{\n    "question": "Hayotimizga kirib kelayotgan yangi ixtirolar va turli texnika ositalari mehnatimizga qanday ta\'sir ko\'rsatadi?",\n    "answer": "Ular mehnatimizni yengillashtirishi va har jabhada bizning imkoniyatlarimizni oshirishi ayni haqiqat."\n}\n]',
 '[\n{\n    "question": "Yurtboshimiz Shavkat Mirziyoyevning kitob mahsulotlarini nashr etish va targ\'atish tizimini rivojlantirishga oid qarori qaysi sanada qabul qilingan?",\n    "answer": "2017-yil 13-sentabrda."\n}\n]',
 '[\n{\n"question": "What is the enduring value of the novel «Mehrobdan chayon»?",\n"answer": "The novel «Mehrobdan chayon» has not lost its artistic value over the years and will not lose it."\n}\n]',
 '[\n{\n    "question": "What qualities should religious leaders and intellectuals possess, especially those in prominent positions, according to the text?",\n    "answer": "They should be patriotic, live with the concerns of the people, and prioritize the nation\'s interests over their own."\n}\n]',
 '[\

In [109]:
instruction_dataset

[{'messages': [{'role': 'system', 'content': 'You are a helpful assistant.'},
   {'role': 'user',
    'content': 'Summarize the following text in uzbek:\n\nADABIYOT MEHROBIDAGI ABADIYAT\nZamonning talabi katta, shiddati esa tez. Bugunning yangi\nkashfiyotidan hayratga to‘la taassurotingiz arimay turib boshga\nbir yangilik sizni o‘z sirliligiga tortishi ham bor gap. Albatta,\nyangi-yangi ixtirolar, hayotimizga kirib kelayotgan turli texnika\nositalari mehnatimizni yengillashtirishi, har jabhada bizning\nimkoniyatlarimizni oshirishi ayni haqigat. Ammo jamiyat hayo-\ntida qanchadan qancha kashfiyotlar qilinmasin inson ma’naviy\ndunyosining rivojida kitobning '},
   {'role': 'assistant',
    'content': "Matn zamonaviy hayotning shiddatli tezligi va doimiy yangi kashfiyotlar, texnik vositalarning hayotni yengillashtirishi hamda imkoniyatlarni oshirishini ta'kidlaydi. Biroq, jamiyatda qancha yangiliklar bo'lishiga qaramay, insonning ma'naviy dunyosi rivojida kitobning o'rni beqiyos ekanligin